# NPS Open Climate Data — Pipeline

Run from the repo root. Two paths:

| | Path A | Path B |
|---|---|---|
| **Data** | Real (DAYMET + ERA5 via Earth Engine tasks) | Synthetic demo data |
| **Requires** | Google Earth Engine project | Nothing extra |
| **Time** | ~2–4 hrs for all 63 parks (serverless) | ~10 seconds |

Run **Step 1** first, then either **Path A** (real data) or **Path B** (demo), then **Steps 3–5**.

## 1. Install dependencies

In [ ]:
import subprocess, sys, os

# Make sure we're running from the repo root
repo_root = os.path.dirname(os.path.abspath("pipeline.ipynb"))
os.chdir(repo_root)
print("Working directory:", os.getcwd())

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".",
     "numpy", "pyarrow", "requests", "google-api-python-client", "--quiet"],
    check=True,
)
print("Done.")

---
## Path A — Real Earth Engine export (serverless, async)

Skip to **Path B** if you just want synthetic demo data.

Export tasks run on Google's servers — you can close the laptop once they're submitted.
Results land in a Google Drive folder; download them when the tasks are done.

### A1. Authenticate with Earth Engine

You need a [Google Cloud project with the Earth Engine API enabled](https://developers.google.com/earth-engine/guides/access).
The first time this runs it opens a browser window to authorize access.

In [ ]:
import ee

GCP_PROJECT = "your-project-id"  # <-- replace with your GCP project

ee.Authenticate()  # opens browser on first run; skips if already authenticated
ee.Initialize(project=GCP_PROJECT)
print("Earth Engine initialized.")

### A2-test. Validate pipeline with Acadia first

Run **this section** before submitting all 63 parks. Acadia is a
single-part CONUS park so it exercises both DAYMET and ERA5. Review
the validation output at the bottom — if anything looks wrong, fix it
before spending the time and quota on the full batch.

**Skip to A2-full** once you're happy with the results.

In [ ]:
from nps_climate_data.batch import submit_park_tasks

DRIVE_FOLDER = "NPS_Climate_Data"  # same folder used in A2-full

acadia_tasks = submit_park_tasks(
    park=next(p for p in __import__('nps_climate_data.parks', fromlist=['get_parks']).get_parks()
              if p['slug'] == 'acadia'),
    drive_folder=DRIVE_FOLDER,
    start="1980-01-01",
)
print(f"Submitted {len(acadia_tasks)} Acadia task(s). Monitor at:")
print("https://code.earthengine.google.com/tasks")

In [ ]:
from nps_climate_data.batch import wait_for_tasks

# Blocks until the task completes (~2–5 min for a single park).
wait_for_tasks(acadia_tasks)

In [ ]:
from nps_climate_data.batch import download_from_drive

download_from_drive(
    drive_folder=DRIVE_FOLDER,
    out_root="data",
    stems=["nps-acadia"],
    quota_project=GCP_PROJECT,
)
print("Downloaded to data/raw/acadia/")

In [ ]:
# Validate the Acadia CSV before running the full batch.
import pandas as pd
from pathlib import Path

csv_path = next(Path("data/raw/acadia").glob("*.csv"))
raw = pd.read_csv(csv_path)
print(f"Raw rows: {len(raw)}  Columns: {len(raw.columns)}")
print()

# 1. Both datasets present
daymet_cols = [c for c in raw.columns if c.startswith("DAYMET_")]
era5_cols   = [c for c in raw.columns if c.startswith("ERA5_")]
assert daymet_cols, "MISSING: no DAYMET_ columns"
assert era5_cols,   "MISSING: no ERA5_ columns"
print(f"DAYMET bands ({len(daymet_cols)}): {daymet_cols}")
print(f"ERA5 bands   ({len(era5_cols)}): {era5_cols}")
print()

# 2. Coalesce duplicate date rows (one DAYMET + one ERA5 per date)
daily = raw.groupby("date").max(numeric_only=True).reset_index()
daily["date"] = pd.to_datetime(daily["date"])
n_dates = len(daily)
date_range = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")
coverage = n_dates / len(date_range)
print(f"Unique dates: {n_dates}  ({daily['date'].min().date()} – {daily['date'].max().date()})")
assert coverage > 0.95, f"Low date coverage: {coverage:.1%}"
print(f"Date coverage: {coverage:.1%}  ✓")
print()

# 3. Plausible value ranges (Acadia, Maine)
checks = [
    ("DAYMET_tmax",                  -40,  50,  "°C"),
    ("DAYMET_tmin",                  -50,  40,  "°C"),
    ("DAYMET_prcp",                    0, 200,  "mm/day"),
    ("ERA5_temperature_2m",          230, 320,  "K"),
    ("ERA5_total_precipitation_sum",   0,   0.3,"m"),
    ("ERA5_snowfall_sum",              0,   0.1,"m w.e."),
    ("ERA5_snowmelt_sum",              0,   0.1,"m w.e."),
    ("ERA5_snow_cover",                0, 100,  "%"),
    ("ERA5_snow_depth",                0,   5,  "m w.e."),
    ("ERA5_potential_evaporation_sum",-0.05, 0, "m (negative)"),
    ("ERA5_total_evaporation_sum",   -0.05, 0,  "m (negative)"),
]
all_ok = True
for col, lo, hi, unit in checks:
    if col not in daily.columns:
        print(f"  MISSING  {col}")
        all_ok = False
        continue
    mn, mx = daily[col].min(), daily[col].max()
    ok = mn >= lo and mx <= hi
    flag = "✓" if ok else "✗ OUT OF RANGE"
    print(f"  {flag}  {col}: [{mn:.4f}, {mx:.4f}] {unit}")
    if not ok:
        all_ok = False
print()
assert all_ok, "Value range checks failed — review output above before running full batch"
print("All checks passed. Ready to run the full batch (A2-full).")

### A2-full. Submit export tasks for all parks

Submits one EE task per park (or per sub-unit for multipart parks like
Saguaro or Channel Islands) and returns immediately. Tasks run serverlessly
on GEE — **no need to keep the notebook open**.

Monitor progress at: https://code.earthengine.google.com/tasks

To export a subset of parks, set `SLUGS` to a list of slugs
(find all slugs in `nps_climate_data/parks.py`).

In [ ]:
from nps_climate_data.batch import submit_all_tasks

DRIVE_FOLDER = "NPS_Climate_Data"  # Drive folder that will be created for you
SLUGS = None  # e.g. ["yellowstone", "glacier"] for a subset

task_infos = submit_all_tasks(
    drive_folder=DRIVE_FOLDER,
    start="1980-01-01",
    slugs=SLUGS,
)
print(f"\nSubmitted {len(task_infos)} tasks. Close the notebook if you like.")
print("Come back to step A3 once the tasks show COMPLETED at the link above.")

### A3. (Optional) Wait for tasks to finish

Skip this cell to close the notebook and check the task page manually.
Run it if you want to poll right here until everything completes.

In [ ]:
from nps_climate_data.batch import wait_for_tasks
wait_for_tasks(task_infos)  # polls every 30 seconds

### A4. Download results from Google Drive

Once all tasks show **COMPLETED**, run this cell to download the CSVs.

**First, run this once in a terminal** (one-time credential setup for Drive):
```
gcloud auth application-default login \
    --scopes=https://www.googleapis.com/auth/drive.readonly,\
             https://www.googleapis.com/auth/cloud-platform
```

In [ ]:
from nps_climate_data.batch import download_from_drive

download_from_drive(
    drive_folder=DRIVE_FOLDER,
    out_root="data",
    quota_project=GCP_PROJECT,
    # stems=["yellowstone", "saguaro_part-0"],  # uncomment for a subset
)
# CSVs are now in data/raw/<slug>/<slug>.csv

---
## Path B — Synthetic demo data

Generates plausible but synthetic climate series for all 63 parks in ~10 seconds.
Good for testing the full pipeline before running the real EE export.

In [ ]:
import subprocess
subprocess.run(["python", "scripts/03_generate_demo_data.py"], check=True)
print("Demo data written to data/raw/")

---
## 3. Build site JSON summaries

Aggregates raw daily series into annual/seasonal summaries, runs Mann–Kendall
trend tests and Theil–Sen slopes, and writes per-park JSON files.

In [ ]:
import subprocess
subprocess.run(["python", "scripts/02_build_site_data.py"], check=True)
print("Site JSON written to data/site/")

## 4. Fetch park boundaries

Pulls real park polygons from the USGS PAD-US web service, then fills
any gaps with circle approximations.

In [ ]:
import subprocess

result = subprocess.run(["python", "scripts/06_fetch_padus_boundaries.py"])
if result.returncode != 0:
    print("USGS fetch failed — falling back to circle approximations.")

subprocess.run(["python", "scripts/05_generate_boundaries.py"], check=True)
print("Boundaries written to site/public/data/boundaries/")

## 5. Copy data into the site

Copies JSON summaries into `site/public/data/` and gzip-compresses the
raw CSVs for the site's download links.

In [ ]:
import gzip, shutil
from pathlib import Path

pub = Path("site/public/data")
(pub / "parks").mkdir(parents=True, exist_ok=True)

# JSON summaries
shutil.copy("data/site/parks.json", pub / "parks.json")
for f in Path("data/site/parks").glob("*.json"):
    shutil.copy(f, pub / "parks" / f.name)

# Gzip raw CSVs
raw_dst = pub / "raw"
raw_dst.mkdir(exist_ok=True)
for slug_dir in Path("data/raw").iterdir():
    if not slug_dir.is_dir():
        continue
    out_dir = raw_dst / slug_dir.name
    out_dir.mkdir(exist_ok=True)
    for csv in slug_dir.glob("*.csv"):
        with open(csv, "rb") as f_in, gzip.open(out_dir / (csv.name + ".gz"), "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

n_json = len(list((pub / "parks").glob("*.json")))
n_csv  = len(list(raw_dst.rglob("*.gz")))
print(f"Copied {n_json} park JSON files and {n_csv} gzipped CSVs.")
print(f"\nNext: cd site && npm install && npm run dev")